In [1]:
import os
import fastf1
import pandas as pd
import psycopg2
import warnings

warnings.filterwarnings('ignore')
if not os.path.exists("cache"):
    os.makedirs("cache")
fastf1.Cache.enable_cache('cache')
session = fastf1.get_session(2023, "Spanish Grand Prix", "R")
session.load()

core           INFO 	Loading data for Spanish Grand Prix - Race [v3.5.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

In [2]:
lap_df = session.laps.copy()
lap_df = lap_df[lap_df['LapTime'].notnull()] 
lap_df = lap_df.reset_index(drop=True)

for col in ['LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']:
    if col in lap_df.columns:
        lap_df[col] = lap_df[col].dt.total_seconds()

def infer_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    else:
        return "TEXT"


table_name = "f1_lap_data"
columns = lap_df.dtypes
sql_columns = ",\n  ".join([f'"{col}" {infer_sql_type(dtype)}' for col, dtype in columns.items()])

create_stmt = f"""
CREATE TABLE IF NOT EXISTS {table_name} (
  {sql_columns}
);
"""
print(create_stmt)


CREATE TABLE IF NOT EXISTS f1_lap_data (
  "Time" TEXT,
  "Driver" TEXT,
  "DriverNumber" TEXT,
  "LapTime" FLOAT,
  "LapNumber" FLOAT,
  "Stint" FLOAT,
  "PitOutTime" TEXT,
  "PitInTime" TEXT,
  "Sector1Time" FLOAT,
  "Sector2Time" FLOAT,
  "Sector3Time" FLOAT,
  "Sector1SessionTime" TEXT,
  "Sector2SessionTime" TEXT,
  "Sector3SessionTime" TEXT,
  "SpeedI1" FLOAT,
  "SpeedI2" FLOAT,
  "SpeedFL" FLOAT,
  "SpeedST" FLOAT,
  "IsPersonalBest" BOOLEAN,
  "Compound" TEXT,
  "TyreLife" FLOAT,
  "FreshTyre" BOOLEAN,
  "Team" TEXT,
  "LapStartTime" TEXT,
  "LapStartDate" TIMESTAMP,
  "TrackStatus" TEXT,
  "Position" FLOAT,
  "Deleted" BOOLEAN,
  "DeletedReason" TEXT,
  "FastF1Generated" BOOLEAN,
  "IsAccurate" BOOLEAN
);



In [3]:
#Connect to PostgreSQL and create table
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="root",  
    host="localhost",
    port="5432"
)
cur = conn.cursor()
cur.execute(create_stmt)
conn.commit()


columns_list = list(lap_df.columns)
placeholders = ', '.join(['%s'] * len(columns_list))
insert_stmt = f"""
INSERT INTO {table_name} ({', '.join(['"{}"'.format(col) for col in columns_list])})
VALUES ({placeholders})
"""

for _, row in lap_df.iterrows():
    row_values = [None if pd.isna(val) else val for val in row]
    cur.execute(insert_stmt, tuple(row_values))

conn.commit()
cur.close()
conn.close()


In [4]:
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="root",
    host="localhost",
    port="5432"
)
df_from_db = pd.read_sql("SELECT * FROM f1_lap_data", conn)
df_from_db

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,01:03:40.107,VER,1,83.935,1.0,1.0,None,None,NaN,32.084,...,True,Red Bull Racing,01:02:15.963,2023-06-04 13:03:15.970,1,1.0,False,,False,False
1,01:05:00.509,VER,1,80.402,2.0,1.0,None,None,24.186,32.088,...,True,Red Bull Racing,01:03:40.107,2023-06-04 13:04:40.114,1,1.0,False,,False,True
2,01:06:21.008,VER,1,80.499,3.0,1.0,None,None,24.167,32.191,...,True,Red Bull Racing,01:05:00.509,2023-06-04 13:06:00.516,1,1.0,False,,False,True
3,01:07:41.354,VER,1,80.346,4.0,1.0,None,None,24.022,32.159,...,True,Red Bull Racing,01:06:21.008,2023-06-04 13:07:21.015,1,1.0,False,,False,True
4,01:09:01.637,VER,1,80.283,5.0,1.0,None,None,24.034,32.213,...,True,Red Bull Racing,01:07:41.354,2023-06-04 13:08:41.361,1,1.0,False,,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5243,02:25:43.001,SAR,2,81.280,61.0,3.0,None,None,24.354,32.587,...,True,Williams,02:24:21.721,2023-06-04 14:25:21.728,1,20.0,False,,False,True
5244,02:27:05.135,SAR,2,82.134,62.0,3.0,None,None,23.675,33.473,...,True,Williams,02:25:43.001,2023-06-04 14:26:43.008,1,20.0,False,,False,True
5245,02:28:25.555,SAR,2,80.420,63.0,3.0,None,None,23.634,32.486,...,True,Williams,02:27:05.135,2023-06-04 14:28:05.142,1,20.0,False,,False,True
5246,02:29:45.535,SAR,2,79.980,64.0,3.0,None,None,23.602,32.127,...,True,Williams,02:28:25.555,2023-06-04 14:29:25.562,1,20.0,False,,False,True
